# 🛡️ TACYOLO — A100 GPU High-Throughput Training & Distillation Pipeline

### **NVIDIA A100-SXM4 (40GB / 80GB) | PyTorch 2.x | Ultralytics YOLO11s**

Configured for your exact Google Drive layout:
- **Zip Dataset Archive**: `/content/drive/MyDrive/datasets.zip`
- **Tiled Dataset Directory**: `/content/drive/MyDrive/TACYOLO/tiles` (or `tiled_batches`)
- **Repository Root**: `/content/drive/MyDrive/YOLO` (or `/content/YOLO`)
- **Checkpoints & Outputs**: `/content/drive/MyDrive/TACYOLO/{trained_weights, weights, runs, exports}`

---

## ⚙️ Cell 1: Mount Google Drive & Enable Ampere A100 Acceleration

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✅ Google Drive successfully mounted at /content/drive")
except Exception as e:
    print(f"ℹ️ Note: {e}")

# 2. Bind to your exact Google Drive folders
DRIVE_ROOT      = Path("/content/drive/MyDrive/TACYOLO")
DRIVE_ZIP       = Path("/content/drive/MyDrive/datasets.zip")
DRIVE_TILES     = DRIVE_ROOT / "tiles"
DRIVE_TILED_B   = DRIVE_ROOT / "tiled_batches"
TRAINED_W_DIR   = DRIVE_ROOT / "trained_weights"
WEIGHTS_DIR     = DRIVE_ROOT / "weights"
RUNS_DIR        = DRIVE_ROOT / "runs"
EXPORTS_DIR     = DRIVE_ROOT / "exports"
METRICS_DIR     = DRIVE_ROOT / "metrics_logs"

for d in [DRIVE_ROOT, TRAINED_W_DIR, WEIGHTS_DIR, RUNS_DIR, EXPORTS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 3. Locate repo from your Drive 'YOLO' folder or /content
DRIVE_REPO = Path("/content/drive/MyDrive/YOLO")
LOCAL_REPO = Path("/content/YOLO")

if DRIVE_REPO.exists():
    REPO_DIR = DRIVE_REPO
    print(f"📁 Using repo from Google Drive: {REPO_DIR}")
elif LOCAL_REPO.exists():
    REPO_DIR = LOCAL_REPO
    print(f"📁 Using repo from /content: {REPO_DIR}")
else:
    REPO_DIR = LOCAL_REPO
    !git clone --depth 1 https://github.com/devansh3108-2/tacyolo.git /content/YOLO

# 4. Activate A100 Ampere TF32 Acceleration
import torch
print(f"\n🖥️ PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🎯 Active GPU: {gpu_name} ({gpu_vram:.1f} GB VRAM)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("🚀 Ampere TF32 & cuDNN Benchmark ENABLED for A100!")
else:
    print("⚠️ Switch runtime to A100 under Runtime -> Change runtime type.")

## 📦 Cell 2: Install High-Performance Dependencies

In [ ]:
%cd {REPO_DIR}
!pip install -q ultralytics ensemble-boxes kagglehub huggingface_hub pyyaml opencv-python-headless
print("\n✅ Dependencies installed!")

## 📥 Cell 3: Load Dataset & Auto-Sanitize `data.yaml` Paths
- Unzips `/content/drive/MyDrive/datasets.zip` or syncs `/content/drive/MyDrive/TACYOLO/tiles` directly to local NVMe (`/content/dataset`)
- **Finds Actual Images** (strictly ignoring `labels/` folders)
- Writes direct **Absolute Image Paths** into `data.yaml` for instant Ultralytics validation

In [ ]:
import os
import yaml
import shutil
from pathlib import Path
from ultralytics import settings

# High-speed local NVMe folder on Colab
LOCAL_DATASET = Path("/content/dataset")
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

DRIVE_ZIP = Path("/content/drive/MyDrive/datasets.zip")
DRIVE_TILES = Path("/content/drive/MyDrive/TACYOLO/tiles")
DRIVE_TILED_BATCHES = Path("/content/drive/MyDrive/TACYOLO/tiled_batches")

dataset_loaded = False

# 1. Unzip /content/drive/MyDrive/datasets.zip directly to NVMe SSD
if DRIVE_ZIP.exists():
    print(f"📦 Found Drive dataset archive: {DRIVE_ZIP} ({DRIVE_ZIP.stat().st_size / (1024**3):.2f} GB)")
    print(f"⚡ Fast unzipping directly to local NVMe ({LOCAL_DATASET})...")
    !unzip -q -o "/content/drive/MyDrive/datasets.zip" -d /content/dataset
    print("✅ datasets.zip successfully extracted to /content/dataset!")
    dataset_loaded = True

# 2. Sync existing tiles from /content/drive/MyDrive/TACYOLO/tiles
elif DRIVE_TILES.exists() and any(DRIVE_TILES.iterdir()):
    print(f"📁 Found pre-generated tiles folder: {DRIVE_TILES}")
    print(f"⚡ Syncing tiles to local NVMe ({LOCAL_DATASET})...")
    !rsync -a --info=progress2 "/content/drive/MyDrive/TACYOLO/tiles/" /content/dataset/
    print("✅ Tiles synced from /content/drive/MyDrive/TACYOLO/tiles!")
    dataset_loaded = True

# 3. Sync existing tiled_batches from /content/drive/MyDrive/TACYOLO/tiled_batches
elif DRIVE_TILED_BATCHES.exists() and any(DRIVE_TILED_BATCHES.iterdir()):
    print(f"📁 Found pre-generated tiled_batches: {DRIVE_TILED_BATCHES}")
    print(f"⚡ Syncing to local NVMe ({LOCAL_DATASET})...")
    !rsync -a --info=progress2 "/content/drive/MyDrive/TACYOLO/tiled_batches/" /content/dataset/
    print("✅ Tiles synced from tiled_batches!")
    dataset_loaded = True

# ─── 4. IMAGE FOLDER RESOLUTION & ABSOLUTE data.yaml ──────────────────────────
settings.update({"datasets_dir": str(LOCAL_DATASET.resolve())})
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Strictly locate train and val folders that contain ACTUAL IMAGES (ignoring labels folders)
train_img_dir = None
val_img_dir = None

for cand in LOCAL_DATASET.rglob("train"):
    if cand.is_dir() and "labels" not in cand.parts:
        if any(f.suffix.lower() in IMG_EXTS for f in cand.glob("*.*" )):
            train_img_dir = cand
            break

for cand in LOCAL_DATASET.rglob("val"):
    if cand.is_dir() and "labels" not in cand.parts:
        if any(f.suffix.lower() in IMG_EXTS for f in cand.glob("*.*" )):
            val_img_dir = cand
            break

if not train_img_dir:
    for cand in LOCAL_DATASET.rglob("images"):
        if (cand / "train").exists():
            train_img_dir = cand / "train"
            val_img_dir = cand / "val"
            break

if not train_img_dir:
    train_img_dir = LOCAL_DATASET / "images" / "train"
    val_img_dir = LOCAL_DATASET / "images" / "val"

if not val_img_dir or not val_img_dir.exists():
    val_img_dir = train_img_dir

if train_img_dir.parent.name == "images":
    actual_dataset_root = train_img_dir.parent.parent
else:
    actual_dataset_root = train_img_dir.parent

names = {
    0: "person", 1: "bird", 2: "drone", 3: "fixed_wing_uav",
    4: "helicopter", 5: "airplane", 6: "tank", 7: "armored_vehicle",
    8: "military_truck", 9: "artillery", 10: "civilian_vehicle", 11: "boat"
}
nc = 12

for y in LOCAL_DATASET.rglob("*.yaml"):
    if "labels" in y.parts:
        continue
    try:
        with open(y, "r", encoding="utf-8") as f:
            yd = yaml.safe_load(f)
            if isinstance(yd, dict) and "names" in yd:
                names = yd["names"]
                nc = yd.get("nc", len(names))
                break
    except Exception:
        pass

# WRITE MASTER data.yaml WITH ABSOLUTE IMAGE PATHS
DATA_YAML = actual_dataset_root / "data.yaml"
master_content = {
    "path": str(actual_dataset_root.resolve().as_posix()),
    "train": str(train_img_dir.resolve().as_posix()),
    "val": str(val_img_dir.resolve().as_posix()),
    "nc": nc,
    "names": names,
}
DATA_YAML.write_text(yaml.dump(master_content, sort_keys=False), encoding="utf-8")

if DATA_YAML != (LOCAL_DATASET / "data.yaml"):
    shutil.copy2(DATA_YAML, LOCAL_DATASET / "data.yaml")

# Fallback symlinks for Ultralytics default search directories
!mkdir -p /content/YOLO/datasets/datasets
!ln -sfn '{actual_dataset_root}' /content/YOLO/datasets/tactical_cuas_armor
!ln -sfn '{actual_dataset_root}' /content/YOLO/datasets/datasets/tactical_cuas_armor
!ln -sfn '{LOCAL_DATASET}' /content/YOLO/datasets/dataset

print(f"\n✅ Master data.yaml validated and saved to: {DATA_YAML}")
print(f"   Train Image Dir: {train_img_dir.resolve()} ({sum(1 for f in train_img_dir.glob('*.*') if f.suffix.lower() in IMG_EXTS)} images)")
print(f"   Val Image Dir  : {val_img_dir.resolve()} ({sum(1 for f in val_img_dir.glob('*.*') if f.suffix.lower() in IMG_EXTS)} images)")
!ls -lh '{actual_dataset_root}'

## 🏋️ Cell 4: Train YOLO11s Student Detector on A100
- **Batch Size**: 64 (saturated for A100 40GB/80GB)
- **RAM Caching**: `cache='ram'` uses Colab High-RAM for zero latency
- **Drive Checkpoints**: Automatically saves to both `trained_weights/` and `weights/`
- **Runs**: Outputs full metric logs directly into `runs/`

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

STUDENT_MODEL = "yolo11s.pt"

# Check for resume from either Drive weights directory
CHECKPOINT_CANDIDATES = [
    TRAINED_W_DIR / "last.pt",
    WEIGHTS_DIR / "last.pt",
    TRAINED_W_DIR / "best.pt",
    WEIGHTS_DIR / "best.pt"
]

starting_weights = STUDENT_MODEL
resume_flag = False
for ckpt in CHECKPOINT_CANDIDATES:
    if ckpt.exists():
        starting_weights = str(ckpt)
        resume_flag = True
        print(f"🔄 Resuming from Drive checkpoint: {ckpt}")
        break

if not resume_flag:
    print(f"🆕 Starting fresh training from: {STUDENT_MODEL}")

model = YOLO(starting_weights)

# Launch training on A100 (label_smoothing removed for Ultralytics 8.4+ compatibility)
results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    batch=64,           # A100 optimal batch size
    imgsz=640,
    device=0,
    workers=16,
    project=str(RUNS_DIR),
    name="a100_tactical_yolo11s",
    exist_ok=True,
    amp=True,
    resume=resume_flag,
    close_mosaic=10,
    cos_lr=True,
    cache="ram",        # High-RAM Colab cache
    patience=25,
    verbose=True
)

# Mirror checkpoints to both trained_weights/ and weights/
run_best = RUNS_DIR / "a100_tactical_yolo11s" / "weights" / "best.pt"
run_last = RUNS_DIR / "a100_tactical_yolo11s" / "weights" / "last.pt"

if run_best.exists():
    for target_dir in [TRAINED_W_DIR, WEIGHTS_DIR]:
        shutil.copy2(run_best, target_dir / "best.pt")
        shutil.copy2(run_best, target_dir / "tactical_yolo11s.pt")
    print("⭐ Best model saved to both 'trained_weights/' and 'weights/' on Drive!")

if run_last.exists():
    for target_dir in [TRAINED_W_DIR, WEIGHTS_DIR]:
        shutil.copy2(run_last, target_dir / "last.pt")
    print("💾 Last checkpoint saved to both 'trained_weights/' and 'weights/' on Drive!")

## ⚡ Cell 5: Export to `exports/` (TensorRT FP16 + ONNX) & Benchmark Latency

In [ ]:
from ultralytics import YOLO
import numpy as np
import torch
import shutil
from pathlib import Path

best_file = WEIGHTS_DIR / "best.pt"
if not best_file.exists():
    best_file = RUNS_DIR / "a100_tactical_yolo11s" / "weights" / "best.pt"

export_model = YOLO(str(best_file))

# 1. Export ONNX (for Jetson deployment)
onnx_path = export_model.export(format="onnx", half=True, imgsz=640, simplify=True)
shutil.copy2(onnx_path, EXPORTS_DIR / Path(onnx_path).name)
print(f"✅ ONNX saved to Drive: {EXPORTS_DIR / Path(onnx_path).name}")

# 2. Export TensorRT FP16 Engine
try:
    trt_engine = export_model.export(format="engine", half=True, imgsz=640, device=0)
    shutil.copy2(trt_engine, EXPORTS_DIR / Path(trt_engine).name)
    print(f"⚡ TensorRT Engine saved to Drive: {EXPORTS_DIR / Path(trt_engine).name}")
except Exception as e:
    print(f"ℹ️ TensorRT compilation note: {e}")

# 3. Run Inference Latency Benchmark
bench_model = YOLO(str(best_file))
dummy_frame = np.zeros((640, 640, 3), dtype=np.uint8)

# Warmup
for _ in range(30):
    _ = bench_model(dummy_frame, device=0, verbose=False)
    torch.cuda.synchronize()

latencies = []
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

for _ in range(150):
    start_event.record()
    _ = bench_model(dummy_frame, device=0, verbose=False)
    end_event.record()
    torch.cuda.synchronize()
    latencies.append(start_event.elapsed_time(end_event))

lat = np.array(latencies)
print("\n" + "="*50)
print("  A100 INFERENCE BENCHMARK (Batch=1, 640x640)")
print("="*50)
print(f"  P50 (Median Latency) : {np.median(lat):.2f} ms")
print(f"  P90 Latency          : {np.percentile(lat, 90):.2f} ms")
print(f"  P99 Latency          : {np.percentile(lat, 99):.2f} ms")
print(f"  Throughput           : {1000.0 / np.mean(lat):.1f} FPS")
print("="*50)